# 03a — Silver: Weather Flatten & Bike-Weather Join
**Bronze weather → Flattened Silver → Joined with bike demand**

Reads the AccuWeather streaming data from **weather_bronze_raw** (landed by Eventstream),
flattens the deeply nested JSON structs, then joins with bike demand data on time (hourly).

### Data Reality
- Weather data streams as **city-level observations** (one `locationName` per snapshot)
- All bike stations share the **same weather** at any given hour
- Join key is **hour** (`dateTime` truncated to hour ↔ `event_hour`)

### Silver Tables Produced
| Silver Table | Description |
|---|---|
| `silver_weather_flattened` | Weather observations with all structs exploded to flat columns |
| `silver_weather_hourly` | Deduplicated hourly weather (one row per hour) |
| `silver_weather_bike_joined` | Weather + bike demand joined on hour + neighbourhood |

### Prerequisites
1. Attach **bicycles_silver** as the default lakehouse
2. Add **weather_bronze_raw** as an additional lakehouse
3. Add **bikerental_bronze_raw** as an additional lakehouse (for station ref)
4. Eventstream for weather should have run at least once
5. Run **NB03 Silver** first so silver_hourly_demand exists

### Weather Schema (AccuWeather via Fabric RTI)
Most columns are nested structs: `{value, unitLabel, unitType}`.
This notebook extracts `.value` from each, plus wind direction/speed sub-structs.

In [ ]:
# ============================================================
# CELL 1 — Configuration · Imports · Read Weather Bronze
# ============================================================

from pyspark.sql.functions import (
    col, lit, when, coalesce, round as spark_round,
    count, countDistinct, avg as spark_avg, sum as spark_sum,
    min as spark_min, max as spark_max, stddev,
    hour, dayofweek, date_format, date_trunc, to_date, to_timestamp,
    row_number, current_timestamp, monotonically_increasing_id,
    concat_ws, sha2, expr,
    year, month, dayofmonth,
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType, DoubleType, LongType, StringType, BooleanType,
    StructType, StructField, TimestampType,
)
from pyspark.sql import SparkSession
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()

# ── Configuration — use fully qualified table names for pipeline execution ──
# In Fabric, schema-enabled lakehouses: {LAKEHOUSE}.dbo.{table}
# Non-schema lakehouses:                {LAKEHOUSE}.{table}
WEATHER_BRONZE_LAKEHOUSE = "weather_bronze_raw"
WEATHER_BRONZE_SCHEMA    = f"{WEATHER_BRONZE_LAKEHOUSE}.dbo"  # Tables under dbo schema

SILVER_LAKEHOUSE = "bicycles_silver"
SILVER_SCHEMA    = f"{SILVER_LAKEHOUSE}"                       # No dbo schema

# ABFSS fallback IDs (if spark_catalog doesn't resolve .dbo)
WORKSPACE_ID          = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"
WEATHER_LAKEHOUSE_ID  = "7a41c5d2-3399-48fe-b581-18be00f36d54"

print(f"Spark version: {spark.version}")
print(f"Processing started at: {datetime.now()}")
print(f"\nSource: {WEATHER_BRONZE_SCHEMA}")
print(f"Target: {SILVER_SCHEMA}")

# ── Verify Bronze tables ──────────────────────────────────────
print(f"\nWeather Bronze tables ({WEATHER_BRONZE_SCHEMA}):")
try:
    spark.sql(f"SHOW TABLES IN {WEATHER_BRONZE_SCHEMA}").show(truncate=False)
except Exception as e:
    print(f"  Could not list tables via catalog: {e}")

# ── Read raw weather ──────────────────────────────────────────
try:
    df_weather_raw = spark.sql(f"SELECT * FROM {WEATHER_BRONZE_SCHEMA}.weather_tb")
    print(f"✓ Reading weather via: {WEATHER_BRONZE_SCHEMA}.weather_tb")
except Exception:
    # Fallback: direct Delta read via ABFSS (schema-enabled lakehouse)
    _abfss = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{WEATHER_LAKEHOUSE_ID}"
    df_weather_raw = spark.read.format("delta").load(f"{_abfss}/Tables/dbo/weather_tb")
    print(f"✓ Reading weather via ABFSS fallback: weather_bronze_raw/dbo/weather_tb")

ROW_COUNT = df_weather_raw.count()
print(f"Weather rows    : {ROW_COUNT:,}")
print(f"Weather columns : {df_weather_raw.columns}")
print(f"Schema:")
df_weather_raw.printSchema()

In [ ]:
# ============================================================
# CELL 2 — Flatten Nested Structs → silver_weather_flattened
# ============================================================
# AccuWeather schema: most columns are STRUCT{value, unitLabel, unitType}
# We extract .value from each, plus wind has nested direction/speed.
# Result: all flat columns, ready for analytics & ML.

df_flat = (
    df_weather_raw
    # ── Parse dateTime string → timestamp ─────────────────────
    .withColumn("observation_time", to_timestamp(col("dateTime")))
    .withColumn("observation_hour", date_trunc("hour", col("observation_time")))
    .withColumn("observation_date", to_date(col("observation_time")))

    # ── Simple flat columns (already primitive) ───────────────
    .withColumn("weather_description",  col("description"))
    .withColumn("has_precipitation",    col("hasPrecipitation"))
    .withColumn("relative_humidity",    col("relativeHumidity").cast(DoubleType()))
    .withColumn("uv_index",            col("uvIndex").cast(IntegerType()))
    .withColumn("uv_description",      col("uvIndexDescription"))
    .withColumn("cloud_cover_pct",     col("cloudCover").cast(IntegerType()))
    .withColumn("obstruction",         col("obstructionsToVisibility"))
    .withColumn("is_daytime",          col("daytime"))

    # ── Extract .value from STRUCT columns ────────────────────
    .withColumn("temperature_c",       col("temperature.value").cast(DoubleType()))
    .withColumn("feels_like_c",        col("realFeelTemperature.value").cast(DoubleType()))
    .withColumn("feels_like_shade_c",  col("realFeelTemperatureShade.value").cast(DoubleType()))
    .withColumn("dew_point_c",         col("dewPoint.value").cast(DoubleType()))
    .withColumn("visibility_km",       col("visibility.value").cast(DoubleType()))
    .withColumn("cloud_ceiling_m",     col("cloudCeiling.value").cast(DoubleType()))
    .withColumn("pressure_mb",         col("pressure.value").cast(DoubleType()))
    .withColumn("pressure_trend",      col("pressureTendency.description"))
    .withColumn("apparent_temp_c",     col("apparentTemperature.value").cast(DoubleType()))
    .withColumn("wind_chill_c",        col("windChillTemperature.value").cast(DoubleType()))
    .withColumn("wet_bulb_c",          col("wetBulbTemperature.value").cast(DoubleType()))
    .withColumn("icon_code",           col("iconCode.value").cast(IntegerType()))

    # ── Wind: nested struct with direction{degrees, description} + speed{value} ──
    .withColumn("wind_speed_kmh",      col("wind.speed.value").cast(DoubleType()))
    .withColumn("wind_direction_deg",  col("wind.direction.degrees").cast(IntegerType()))
    .withColumn("wind_direction_desc", col("wind.direction.description"))
    .withColumn("wind_gust_kmh",       col("windGust.speed.value").cast(DoubleType()))

    # ── Precipitation summary (deep nesting) ──────────────────
    .withColumn("precip_past_1h_mm",   col("precipitationSummary.pastHour.value").cast(DoubleType()))
    .withColumn("precip_past_3h_mm",   col("precipitationSummary.past3Hours.value").cast(DoubleType()))
    .withColumn("precip_past_6h_mm",   col("precipitationSummary.past6Hours.value").cast(DoubleType()))
    .withColumn("precip_past_12h_mm",  col("precipitationSummary.past12Hours.value").cast(DoubleType()))
    .withColumn("precip_past_24h_mm",  col("precipitationSummary.past24Hours.value").cast(DoubleType()))

    # ── Temperature summary ───────────────────────────────────
    .withColumn("temp_min_6h_c",       col("temperatureSummary.past6Hours.minimum.value").cast(DoubleType()))
    .withColumn("temp_max_6h_c",       col("temperatureSummary.past6Hours.maximum.value").cast(DoubleType()))
    .withColumn("temp_min_24h_c",      col("temperatureSummary.past24Hours.minimum.value").cast(DoubleType()))
    .withColumn("temp_max_24h_c",      col("temperatureSummary.past24Hours.maximum.value").cast(DoubleType()))

    # ── Location ──────────────────────────────────────────────
    .withColumn("weather_latitude",    col("location.latitude").cast(DoubleType()))
    .withColumn("weather_longitude",   col("location.longitude").cast(DoubleType()))
    .withColumn("location_name",       col("locationName"))

    # ── Select only the flat columns ──────────────────────────
    .select(
        "observation_time", "observation_hour", "observation_date",
        "weather_description", "has_precipitation", "icon_code",
        "temperature_c", "feels_like_c", "feels_like_shade_c",
        "relative_humidity", "dew_point_c",
        "wind_speed_kmh", "wind_direction_deg", "wind_direction_desc", "wind_gust_kmh",
        "uv_index", "uv_description",
        "visibility_km", "obstruction", "cloud_cover_pct", "cloud_ceiling_m",
        "pressure_mb", "pressure_trend",
        "apparent_temp_c", "wind_chill_c", "wet_bulb_c",
        "precip_past_1h_mm", "precip_past_3h_mm", "precip_past_6h_mm",
        "precip_past_12h_mm", "precip_past_24h_mm",
        "temp_min_6h_c", "temp_max_6h_c", "temp_min_24h_c", "temp_max_24h_c",
        "is_daytime", "weather_latitude", "weather_longitude", "location_name",
    )
    .withColumn("processed_at", current_timestamp())
)

df_flat.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER_LAKEHOUSE}.silver_weather_flattened")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.silver_weather_flattened").collect()[0]["c"]
print(f"✓ silver_weather_flattened: {_cnt:,} rows  ({len(df_flat.columns)} columns)")
print(f"  Location(s): {[r[0] for r in df_flat.select('location_name').distinct().collect()]}")

# Show sample
df_flat.select(
    "observation_time", "weather_description", "temperature_c",
    "wind_speed_kmh", "has_precipitation", "cloud_cover_pct",
).show(5, truncate=False)

In [ ]:
# ============================================================
# CELL 3 — Hourly Weather Dedup → silver_weather_hourly
# ============================================================
# Weather may have multiple observations per hour (e.g. every 15 min).
# We pick the latest observation per hour and add derived features
# useful for demand modelling.

w_dedup = Window.partitionBy("observation_hour").orderBy(col("observation_time").desc())

df_hourly_weather = (
    df_flat
    .withColumn("_rn", row_number().over(w_dedup))
    .filter(col("_rn") == 1)
    .drop("_rn")

    # ── Derived weather features for ML & analytics ───────────
    .withColumn(
        "weather_severity",
        when(col("wind_speed_kmh") > 50, "severe")          # storm-force
        .when(col("has_precipitation") & (col("temperature_c") < 2), "severe")  # ice/snow
        .when(col("has_precipitation") & (col("wind_speed_kmh") > 30), "bad")
        .when(col("has_precipitation"), "moderate")
        .when(col("wind_speed_kmh") > 30, "moderate")
        .when(col("cloud_cover_pct") > 80, "fair")
        .otherwise("good"),
    )
    .withColumn(
        "cycling_comfort_index",
        # Score 0–100: higher = better for cycling
        spark_round(
            lit(100)
            - when(col("has_precipitation"), lit(30)).otherwise(lit(0))        # rain penalty
            - when(col("wind_speed_kmh") > 25, lit(15))                       # wind penalty
                .when(col("wind_speed_kmh") > 15, lit(8))
                .otherwise(lit(0))
            - when(col("temperature_c") < 5, lit(20))                         # cold penalty
                .when(col("temperature_c") < 10, lit(10))
                .when(col("temperature_c") > 35, lit(15))                     # heat penalty
                .when(col("temperature_c") > 30, lit(8))
                .otherwise(lit(0))
            - when(col("visibility_km") < 2, lit(15))                         # fog penalty
                .when(col("visibility_km") < 5, lit(5))
                .otherwise(lit(0))
            - when(col("cloud_cover_pct") > 90, lit(5)).otherwise(lit(0)),    # overcast
            0,
        ),
    )
    .withColumn(
        "weather_category",
        when(col("icon_code").isin(1, 2, 3, 4, 5), "Sunny/Clear")
        .when(col("icon_code").isin(6, 7, 8), "Mostly Cloudy")
        .when(col("icon_code").isin(11, 37, 38), "Fog")
        .when(col("icon_code").isin(12, 13, 14, 18, 39, 40), "Rain")
        .when(col("icon_code").isin(15, 16, 17, 41, 42), "Thunderstorm")
        .when(col("icon_code").isin(19, 20, 21, 22, 23, 24, 25, 26, 29, 43, 44), "Snow/Ice")
        .otherwise("Other"),
    )
    .withColumn("hour_of_day", hour("observation_hour"))
    .withColumn("day_of_week", dayofweek("observation_hour"))
    .withColumn("processed_at", current_timestamp())
)

df_hourly_weather.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER_LAKEHOUSE}.silver_weather_hourly")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.silver_weather_hourly").collect()[0]["c"]
print(f"✓ silver_weather_hourly: {_cnt:,} rows (1 per hour)")

# Show severity distribution
df_hourly_weather.groupBy("weather_severity").count().orderBy("count", ascending=False).show()
df_hourly_weather.groupBy("weather_category").count().orderBy("count", ascending=False).show()

In [ ]:
# ============================================================
# CELL 4 — Join Weather + Bike Demand → silver_weather_bike_joined
# ============================================================
# Weather is city-level (one observation per hour).
# Bike demand is per neighbourhood per hour.
# Join: weather_hourly.observation_hour = hourly_demand.event_hour
# Result: every (neighbourhood, hour) demand row gets weather context.

df_demand = spark.sql(f"SELECT * FROM {SILVER_LAKEHOUSE}.silver_hourly_demand")
df_weather = spark.sql(f"SELECT * FROM {SILVER_LAKEHOUSE}.silver_weather_hourly")

print(f"Demand rows:  {df_demand.count():,}")
print(f"Weather rows: {df_weather.count():,}")

# Select weather columns for the join (avoid column name collisions)
weather_join_cols = df_weather.select(
    col("observation_hour").alias("_weather_hour"),

    # Core weather metrics
    col("weather_description"),
    col("weather_category"),
    col("weather_severity"),
    col("has_precipitation"),
    col("temperature_c"),
    col("feels_like_c"),
    col("relative_humidity"),
    col("wind_speed_kmh"),
    col("wind_gust_kmh"),
    col("wind_direction_desc"),
    col("visibility_km"),
    col("cloud_cover_pct"),
    col("pressure_mb"),
    col("uv_index"),
    col("is_daytime"),

    # Precipitation accumulation
    col("precip_past_1h_mm"),

    # Derived features
    col("cycling_comfort_index"),

    # Location (for map visuals)
    col("weather_latitude"),
    col("weather_longitude"),
    col("location_name"),
)

df_joined = (
    df_demand
    .join(
        weather_join_cols,
        df_demand["event_hour"] == weather_join_cols["_weather_hour"],
        "left",  # keep demand even if weather is missing
    )
    .drop("_weather_hour")
    .withColumn(
        "weather_demand_impact",
        # Estimate: how much does weather suppress/boost demand?
        # Baseline = 1.0 (neutral), < 1 = suppressed, > 1 = boosted
        when(col("weather_severity") == "severe", lit(0.3))
        .when(col("weather_severity") == "bad", lit(0.5))
        .when(col("weather_severity") == "moderate", lit(0.7))
        .when(col("weather_severity") == "fair", lit(0.9))
        .when(col("weather_severity") == "good", lit(1.0))
        .otherwise(lit(0.8)),  # unknown → conservative
    )
    .withColumn(
        "weather_adjusted_demand",
        spark_round(col("event_count") * col("weather_demand_impact"), 1),
    )
    .withColumn("processed_at", current_timestamp())
)

df_joined.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER_LAKEHOUSE}.silver_weather_bike_joined")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.silver_weather_bike_joined").collect()[0]["c"]
_matched = df_joined.filter(col("temperature_c").isNotNull()).count()
_unmatched = df_joined.filter(col("temperature_c").isNull()).count()

print(f"\n✓ silver_weather_bike_joined: {_cnt:,} rows")
print(f"  Weather matched : {_matched:,}")
print(f"  No weather data : {_unmatched:,}")

# Show sample of the joined data
df_joined.select(
    "event_hour", "Neighbourhood", "event_count",
    "weather_description", "temperature_c", "wind_speed_kmh",
    "cycling_comfort_index", "weather_demand_impact", "weather_adjusted_demand",
).show(10, truncate=False)

In [ ]:
# ============================================================
# CELL 5 — Weather Impact Analysis & Summary
# ============================================================

print("=" * 60)
print("  SILVER WEATHER LAYER — SUMMARY")
print("=" * 60)

tables = {
    "silver_weather_flattened":    "All weather obs, flat columns",
    "silver_weather_hourly":       "Deduped hourly (1 per hour)",
    "silver_weather_bike_joined":  "Demand + weather joined",
}
print(f"\n{'Table':<40} {'Rows':>10}  Description")
print("-" * 75)
for tbl, desc in tables.items():
    try:
        n = spark.sql(f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.{tbl}").collect()[0]["c"]
        print(f"  {tbl:<38} {n:>10,}  {desc}")
    except Exception:
        print(f"  {tbl:<38} {'MISSING':>10}  {desc}")

# ── Weather impact preview ────────────────────────────────────
print("\n── Weather Impact on Demand ─────────────────────────")
df_impact = spark.sql(f"""
    SELECT
        weather_severity,
        COUNT(*) AS observation_count,
        ROUND(AVG(event_count), 1) AS avg_raw_demand,
        ROUND(AVG(weather_adjusted_demand), 1) AS avg_adj_demand,
        ROUND(AVG(weather_demand_impact), 2) AS avg_impact_factor,
        ROUND(AVG(cycling_comfort_index), 0) AS avg_comfort
    FROM {SILVER_LAKEHOUSE}.silver_weather_bike_joined
    WHERE weather_severity IS NOT NULL
    GROUP BY weather_severity
    ORDER BY avg_impact_factor ASC
""")
df_impact.show(truncate=False)

# ── Cycling comfort by weather category ──────────────────────
print("── Cycling Comfort by Weather Type ──────────────────")
df_comfort = spark.sql(f"""
    SELECT
        weather_category,
        COUNT(*) AS hours,
        ROUND(AVG(cycling_comfort_index), 0) AS avg_comfort,
        ROUND(AVG(temperature_c), 1) AS avg_temp,
        ROUND(AVG(wind_speed_kmh), 1) AS avg_wind
    FROM {SILVER_LAKEHOUSE}.silver_weather_bike_joined
    WHERE weather_category IS NOT NULL
    GROUP BY weather_category
    ORDER BY avg_comfort DESC
""")
df_comfort.show(truncate=False)

print("\n✅ NB03a complete — Weather flattened, hourly deduped, bike-weather joined.")
print("   Next: Run NB04 (Gold Star Schema) — now includes dim_weather + fact_weather_impact")